# 📓 Notebook 17 — Feature Engineering

> **Module:** Machine Learning · **Estimated time:** 55–70 min · **Difficulty:** Intermediate

> *"Most of what looks like a model problem is actually a feature problem."* — every senior ML engineer, eventually.

A model can only learn what the features tell it. Better features almost always beat a fancier model — and they are a *much* cheaper investment. This notebook walks through the feature-engineering toolkit on a small tabular dataset, with a special focus on the single most damaging mistake: **target leakage**.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Choose the right **encoding** for categorical features (one-hot, ordinal, target encoding — and when each is wrong).
2. Apply **scaling** correctly (when it matters, when it doesn't, where the leak hides).
3. Extract useful information from **datetime** columns.
4. Build **interaction** and **ratio** features that lift linear models.
5. Spot and prevent **target leakage** before it ships.
6. Use **`SelectKBest`** and feature importance to prune features.
7. Assemble a complete leak-free **pipeline** that's ready for cross-validation.

## ✅ Prerequisites

NB 15 (sklearn basics) and NB 16 (model evaluation — especially the Pipeline discipline).

## 1. Setup — a slightly richer dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
RANDOM_STATE = 42

# A richer churn dataset that includes a signup_date so we can show date features
rng = np.random.default_rng(RANDOM_STATE)
n = 600
mrr_eur         = rng.lognormal(mean=6.5, sigma=0.7, size=n).clip(50, 50_000)
months_active   = rng.integers(1, 60, size=n)
support_tickets = rng.poisson(lam=2.5, size=n)
plan            = rng.choice(["Free","Standard","Premium","Enterprise"],
                              size=n, p=[0.15,0.45,0.30,0.10])
last_login_days = rng.integers(0, 90, size=n)
signup_date     = pd.to_datetime("2023-01-01") + pd.to_timedelta(
    rng.integers(0, 365, size=n), unit="D")
region          = rng.choice(["EU", "US", "APAC", "LATAM"], size=n, p=[0.4, 0.4, 0.15, 0.05])

plan_adj = np.array([{"Free":-0.45,"Standard":0.0,"Premium":0.15,"Enterprise":0.35}[p] for p in plan])
happiness = (-0.40*(support_tickets-2.5)/2.0 -0.80*(last_login_days-45)/30.0
             +0.35*(np.log1p(mrr_eur)-6.5)/1.5 +0.30*(months_active-30)/20.0
             + plan_adj + rng.normal(0,0.30,n))
churn_prob = 1/(1+np.exp(-(-1.6 - 2.5*happiness)))
churned = (rng.random(n) < churn_prob).astype(int)

df = pd.DataFrame({"mrr_eur":mrr_eur, "months_active":months_active,
                    "support_tickets":support_tickets, "plan":plan,
                    "last_login_days":last_login_days, "signup_date":signup_date,
                    "region":region, "churned":churned})
print(df.head())
print(f"\nShape: {df.shape}, churn rate: {df['churned'].mean():.1%}")


## 2. Categorical encoding — three options, three trade-offs

| Encoding | When to use | When NOT to use |
|---|---|---|
| **One-hot** | Few categories, no ordering | Hundreds of categories — explodes dimensionality |
| **Ordinal** | True ordering (`small/medium/large`) | Nominal categories — model will infer false ordering |
| **Target / mean encoding** | Many categories, you have enough data | Small data — leaks the target value into features |

Let's see each.

In [ ]:
# (a) One-hot — safe default for nominal categorical with few values
oh = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_oh = oh.fit_transform(df[["plan"]])
print("One-hot encoding of 'plan':")
print(pd.DataFrame(X_oh, columns=oh.get_feature_names_out(["plan"])).head())


In [ ]:
# (b) Ordinal — when there IS an order
plan_order = [["Free", "Standard", "Premium", "Enterprise"]]
ord_enc = OrdinalEncoder(categories=plan_order)
df["plan_ordinal"] = ord_enc.fit_transform(df[["plan"]]).astype(int)
print(df[["plan", "plan_ordinal"]].drop_duplicates().sort_values("plan_ordinal"))


> ⚠️ **The classic ordinal mistake.** A column with `["red", "green", "blue"]` is *not* ordinal — `green` is not "between" red and blue. Using ordinal encoding on nominal data injects a fictitious ordering the model will dutifully try to learn.

In [ ]:
# (c) Target encoding — DANGEROUS without care
# Goal: replace each plan with the mean churn rate for that plan (on training data only!)

def target_encode(train_df, valid_df, col, target):
    """Compute target encoding on training data only; apply to validation."""
    means = train_df.groupby(col)[target].mean()
    return train_df[col].map(means), valid_df[col].map(means)


train_df, valid_df = train_test_split(df, test_size=0.25, random_state=RANDOM_STATE,
                                      stratify=df["churned"])
te_train, te_valid = target_encode(train_df, valid_df, "plan", "churned")
print("Target encoding learned on training data:")
print(train_df.groupby("plan")["churned"].mean().round(3))
print("\nFirst 5 encoded training values:")
print(te_train.head())


> 🎯 **Why target encoding leaks if you're sloppy.** If you compute the mean on *all* the data (including future test rows), the test rows see their own target through the encoding. The CV score then looks fantastic — and production accuracy collapses.
>
> **Fix:** compute target encoding inside a CV fold, on training rows only. The `sklearn-contrib` package `category_encoders` does this safely via `TargetEncoder(cv=5)`.

## 3. Scaling — when it matters, when it doesn't

| Model family | Needs scaling? | Why |
|---|---|---|
| Logistic Regression, SVM, k-NN, PCA, neural nets | **Yes** | Distance / gradient-based: unscaled features dominate |
| Decision Tree, Random Forest, Gradient Boosting | **No** | Tree splits are invariant to monotone transformations |
| Naive Bayes | Depends | Discretisation-friendly variants don't need it |

For our churn data, `mrr_eur` ranges 50–50,000 while `support_tickets` is 0–10. A logistic regression sees `mrr_eur` as ~1000× more important *just because of scale*.

In [ ]:
# Demonstrate why scaling matters for LR but not for RF
NUM = ["mrr_eur", "months_active", "support_tickets", "last_login_days"]
X = df[NUM]
y = df["churned"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                           random_state=RANDOM_STATE, stratify=y)

# LR unscaled
lr_raw = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
# LR scaled
lr_scaled = Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
# RF (no scaling needed)
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE).fit(X_tr, y_tr)

print(f"LR no scaling : test score = {lr_raw.score(X_te, y_te):.3f}")
print(f"LR with scaler: test score = {lr_scaled.score(X_te, y_te):.3f}")
print(f"RF (no scale) : test score = {rf.score(X_te, y_te):.3f}")


**Three scaling rules:**

1. **Use `StandardScaler`** by default — zero mean, unit variance.
2. **Use `MinMaxScaler`** when you have a known bounded range or feed images / pixels.
3. **Put the scaler inside the `Pipeline`** — never `fit_transform(X)` on the whole dataset before splitting. (See NB 16 for the leak pattern.)

## 4. Datetime features — turning a date into many signals

A `signup_date` column on its own is useless to a model. But it contains many useful features hiding inside it: day of week, month, year, tenure, season, business-day-or-not.

In [ ]:
# Build a bunch of useful features from a single datetime column
ref = pd.Timestamp("2024-06-01")   # the "as-of" date — usually "now" in production

dfx = df.copy()
dfx["signup_year"]   = dfx["signup_date"].dt.year
dfx["signup_month"]  = dfx["signup_date"].dt.month
dfx["signup_dow"]    = dfx["signup_date"].dt.dayofweek         # 0 = Monday
dfx["signup_q"]      = dfx["signup_date"].dt.quarter
dfx["tenure_days"]   = (ref - dfx["signup_date"]).dt.days
dfx["is_weekend_signup"] = (dfx["signup_dow"] >= 5).astype(int)

# Cyclical encoding for month (so December and January end up close together)
dfx["signup_month_sin"] = np.sin(2*np.pi*dfx["signup_month"]/12)
dfx["signup_month_cos"] = np.cos(2*np.pi*dfx["signup_month"]/12)

print(dfx[["signup_date", "signup_year", "signup_month", "tenure_days",
            "signup_dow", "is_weekend_signup",
            "signup_month_sin", "signup_month_cos"]].head())


**Why cyclical encoding?** A linear model treats month=1 and month=12 as 11 apart — but they're really 1 apart in cyclic time. Sin / cos pairs encode "circular distance" correctly. Same trick works for hours of the day, days of the week, and any periodic phenomenon.

> 💡 **Tenure features are almost always useful.** `tenure_days` (days since signup) is a strong churn predictor on its own — even when you have rich behavioural features.

## 5. Interaction and ratio features — help linear models see non-linearity

In [ ]:
# Two examples:
# (a) Ratio: tickets per month of tenure -- normalises by how long they've been a customer
dfx["tickets_per_month"] = dfx["support_tickets"] / dfx["months_active"].clip(lower=1)

# (b) Interaction: is this a Free-plan customer with low engagement?
dfx["free_and_inactive"] = ((dfx["plan"] == "Free") & (dfx["last_login_days"] > 30)).astype(int)

# (c) Log of a heavy-tailed feature -- spreads the values more linearly
dfx["log_mrr"] = np.log1p(dfx["mrr_eur"])

print(dfx[["mrr_eur", "log_mrr", "support_tickets", "months_active",
            "tickets_per_month", "free_and_inactive"]].head())


> 🎯 **Heavy-tailed features (MRR, latency, follower counts) almost always benefit from `log1p`.** It pulls the long right tail toward the centre so models can use it linearly.

## 6. Target leakage — the bug that ruins ML projects

A feature is **leaky** when it contains information that *wouldn't be available at prediction time*. Three common ways leakage sneaks in:

1. **The label disguised as a feature.** `did_customer_call_to_cancel` predicting `churned` perfectly — because it's basically the label.
2. **Future information.** `total_lifetime_spend` includes spend that happened *after* the churn event.
3. **Group statistics computed on the whole dataset.** `avg_revenue_for_this_plan` computed on train+test together leaks test means.

Let's simulate a subtle leak and watch it produce suspiciously good metrics.

In [ ]:
# Build a "leaky" feature that's slightly informed by the future
# Pretend: total_engagement is a feature we compute, but it secretly used some
# post-event activity. So it correlates with churned more strongly than it should.
np.random.seed(0)
df["total_engagement"] = (
    50 * (1 - df["churned"])            # ← leak: directly uses the target
    + 5 * df["months_active"]
    + np.random.normal(0, 5, len(df))
)

X_leak = df[["mrr_eur", "months_active", "support_tickets", "total_engagement"]]
X_clean = df[["mrr_eur", "months_active", "support_tickets"]]

pipe = Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=1000))])
print(f"With leak     : {cross_val_score(pipe, X_leak,  df['churned'], cv=5).mean():.3f}")
print(f"Without leak  : {cross_val_score(pipe, X_clean, df['churned'], cv=5).mean():.3f}")


**That's the smell.** A new feature pushes CV accuracy by 10+ points with no extra data and no model change → almost certainly a leak.

**Checklist before trusting a "great" feature:**

- Could this value have been measured *before* the event you're predicting?
- Is this an aggregate computed on the whole dataset (instead of training only)?
- Does this column rank-correlate with the target above 0.9 by itself?

If any answer is "yes / I'm not sure", investigate before you ship.

## 7. Feature selection — keeping only what helps

In [ ]:
# Univariate F-statistic ranking of numerical features against churn
NUM = ["mrr_eur", "log_mrr", "months_active", "support_tickets",
       "last_login_days", "tickets_per_month"]
sel = SelectKBest(score_func=f_classif, k="all").fit(dfx[NUM], dfx["churned"])

rank = pd.DataFrame({"feature": NUM,
                      "F_score": sel.scores_,
                      "p_value": sel.pvalues_}).sort_values("F_score", ascending=False)
print(rank.round(4))


**Two practical takeaways:**

- High F-score + low p-value = feature carries signal *on its own*.
- A low score doesn't mean "drop it" — it might still be useful **in combination** with other features. Tree-based models exploit interactions you can't see from a univariate test.

> 💡 **Random-forest feature importance** (`rf.feature_importances_`) is a complementary view: it tells you which features the model *actually used*, including interactions.

## 8. A complete leak-free pipeline

In [ ]:
# All preprocessing inside a single Pipeline / ColumnTransformer
NUM_FEATURES = ["mrr_eur", "log_mrr", "months_active", "support_tickets",
                 "last_login_days", "tickets_per_month", "tenure_days",
                 "signup_month_sin", "signup_month_cos", "is_weekend_signup"]
CAT_FEATURES = ["plan", "region"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
], remainder="drop")

pipe = Pipeline([("prep", preprocess),
                  ("model", LogisticRegression(max_iter=1000))])

# Cross-validation with stratification
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(pipe, dfx, dfx["churned"], cv=cv, scoring="roc_auc")
print(f"5-fold ROC AUC: {scores.round(3)}")
print(f"Mean ± std   : {scores.mean():.3f} ± {scores.std():.3f}")


**Read this pipeline one more time.** Every transformer is fit on each CV fold's training portion only, then applied to that fold's validation portion. There is no path through which validation data could influence a transformer's parameters. *This is what "no leakage" means in code.*

## 🧪 Practice exercises

### Exercise 1 — Add a region-specific tenure feature

Build a new feature `tenure_months` (`tenure_days / 30`) and a categorical feature `is_long_tenure` (`tenure_months > 12`). Add them to the pipeline and report whether they improve the CV score.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
dfx["tenure_months"]  = dfx["tenure_days"] / 30
dfx["is_long_tenure"] = (dfx["tenure_months"] > 12).astype(int)

NEW_NUM = NUM_FEATURES + ["tenure_months", "is_long_tenure"]
prep2 = ColumnTransformer([
    ("num", StandardScaler(), NEW_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
])
pipe2 = Pipeline([("prep", prep2), ("model", LogisticRegression(max_iter=1000))])
scores2 = cross_val_score(pipe2, dfx, dfx["churned"], cv=cv, scoring="roc_auc")
print(f"Original: {scores.mean():.3f} ± {scores.std():.3f}")
print(f"+ tenure: {scores2.mean():.3f} ± {scores2.std():.3f}")
```

If the lift is within the standard deviation, it's noise. The honest rule: a feature should beat the noise by at least 2× the std before you keep it.
</details>

### Exercise 2 — Spot the leak

Look at this proposed feature for predicting churn:

```python
df["avg_churn_rate_for_plan"] = df.groupby("plan")["churned"].transform("mean")
```

Is it leaky? If yes, fix it.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

**Yes — leaky.** `groupby("plan").transform("mean")` is computed over the *entire* DataFrame, including the test set's rows. The feature carries information from the target back into the inputs.

The fix is to compute it on training data only, then apply it as a lookup:

```python
train_means = train_df.groupby("plan")["churned"].mean()
train_df["avg_churn_for_plan"] = train_df["plan"].map(train_means)
valid_df["avg_churn_for_plan"] = valid_df["plan"].map(train_means)
```

Or — much safer — use `category_encoders.TargetEncoder(cv=5)` inside a Pipeline, which does the right thing on each CV fold.
</details>

### Exercise 3 — A custom feature transformer

Write a tiny `LogTransformer` class that follows the sklearn estimator API (`fit`, `transform`) and applies `log1p` to the columns it's given. Plug it into the pipeline in place of the manual `log_mrr` computation.

In [ ]:
# Your code here  👇
from sklearn.base import BaseEstimator, TransformerMixin


<details>
<summary>💡 <b>Solution</b></summary>

```python
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class LogTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p to every column. Stateless; fit is a no-op."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.log1p(np.asarray(X))


prep_log = ColumnTransformer([
    ("log", Pipeline([("log", LogTransformer()), ("s", StandardScaler())]), ["mrr_eur"]),
    ("num", StandardScaler(), [c for c in NUM_FEATURES if c != "log_mrr"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
])
pipe_log = Pipeline([("prep", prep_log), ("model", LogisticRegression(max_iter=1000))])
print(cross_val_score(pipe_log, dfx, dfx["churned"], cv=cv, scoring="roc_auc").mean().round(3))
```

Writing custom transformers is how you keep complex preprocessing inside the
Pipeline. Once it's there, CV stays honest no matter how clever the transformation gets.
</details>

## 🎁 Bonus mini-project — A feature-engineering bake-off

Build a function `bake_off(feature_sets, X, y)` that takes a dict like `{"baseline": [...], "+ time": [...]}` and returns a DataFrame of mean ± std ROC AUC for each feature set. Use it to quantify the marginal value of each feature group.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def bake_off(feature_sets: dict, dfx, target, cv=cv):
    rows = []
    for name, num_cols in feature_sets.items():
        prep = ColumnTransformer([
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
        ])
        pipe = Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=1000))])
        s = cross_val_score(pipe, dfx, dfx[target], cv=cv, scoring="roc_auc")
        rows.append({"feature_set": name,
                     "mean": s.mean(), "std": s.std(),
                     "n_features": len(num_cols) + 2})
    return pd.DataFrame(rows).set_index("feature_set").round(3)


feature_sets = {
    "baseline":       ["mrr_eur", "months_active", "support_tickets", "last_login_days"],
    "+ log_mrr":      ["mrr_eur", "log_mrr", "months_active", "support_tickets", "last_login_days"],
    "+ tenure":       ["mrr_eur", "log_mrr", "months_active", "support_tickets", "last_login_days", "tenure_days"],
    "+ time cyclic":  NUM_FEATURES,
}
print(bake_off(feature_sets, dfx, "churned"))
```

You get a one-line story per feature group: "adding cyclic time features lifts mean AUC by X with a Y-standard-deviation gain". Now you can argue for or against each one.
</details>

## 🧠 Key takeaways

1. **Better features beat fancier models** — most of the time, more cheaply.
2. **Categorical encoding** has three options; target encoding leaks unless you're careful.
3. **Scaling** matters for distance- and gradient-based models, not for trees.
4. **Datetime columns** are gold: extract dow, month, tenure, cyclic features.
5. **Heavy-tailed features** almost always benefit from `log1p`.
6. **Target leakage** is the single most common cause of "too-good-to-be-true" CV scores. Always question a feature that jumps your score by 10+ points.
7. **Wrap all preprocessing in a `Pipeline`** so CV is leak-proof by construction.
8. Run a **feature bake-off** to quantify each feature group's marginal value.

## ✅ Self-assessment

- [ ] Choose one-hot vs ordinal encoding for a column
- [ ] Apply a `StandardScaler` and explain when it does (and doesn't) help
- [ ] Extract 5+ features from a datetime column, including cyclic encoding
- [ ] Spot a leaky feature and propose a leak-free alternative
- [ ] Build a `ColumnTransformer` that combines numeric and categorical preprocessing
- [ ] Write a custom sklearn-compatible transformer
- [ ] Quantify the marginal value of a feature group with cross-validation

## 🚀 Next step

You've now completed Module 4. Continue with **Module 5 — AI Engineering**, where the metrics and discipline you just internalised get applied to a completely different kind of model.